# Evaluation du première approche NEL implémentée pour Décidon, il s'agit de celle qui a servi la pré-annotation du GT.

Cette approche se focalise sur la selection des candidats et un peu moins sur la désambiguisation.

## préparation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import re

# Fichier CSV séparé par des tabulations
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/data/DECIDON/GT_analysis/2026-07-01-CHANDRA-GT-NEL-SPK_PER-linked_linked_consolidated_intra-seance - nel_consolidated_intra.tsv", sep="\t")


/tmp/ipykernel_1181/326183673.py:6: DtypeWarning: Columns (60,71,93,104,115,126,147,148,150,151,152,153,158,159,161,162,163,164) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/data/DECIDON/GT_analysis/2026-07-01-CHANDRA-GT-NEL-SPK_PER-linked_linked_consolidated_intra-seance - nel_consolidated_intra.tsv", sep="\t")


In [ ]:
colonnes_affichage_select = ["entity", "classe", "wikidata_url_valide", "persons_1_wikidata_url", "persons_2_wikidata_url", "persons_3_wikidata_url"]

In [ ]:
df[colonnes_affichage_select].sample(20, random_state=42)

,entity,classe,wikidata_url_valide,persons_1_wikidata_url,persons_2_wikidata_url,persons_3_wikidata_url
3804,Sal (Léonce de),PER,https://www.wikidata.org/wiki/Q3271636,https://www.wikidata.org/wiki/Q3271636,NaN,NaN
4679,Léon Say,PER,https://www.wikidata.org/wiki/Q3271322,https://www.wikidata.org/wiki/Q3271322,https://www.wikidata.org/wiki/Q3271322,NaN
805,le président,SPK,https://www.wikidata.org/wiki/Q471311,https://www.wikidata.org/wiki/Q317957,https://www.wikidata.org/wiki/Q585025,https://www.wikidata.org/wiki/Q471311
3996,Bizot,PER,NIL,https://www.wikidata.org/wiki/Q3059731,NaN,NaN
3523,le Président de la République,PER,https://www.wikidata.org/wiki/Q169502,https://www.wikidata.org/wiki/Q275020,https://www.wikidata.org/wiki/Q296673,https://www.wikidata.org/wiki/Q669725
3651,Cuvinot,PER,https://www.wikidata.org/wiki/Q2415176,https://www.wikidata.org/wiki/Q2415176,https://www.wikidata.org/wiki/Q2415176,NaN
2416,Emile Faure,SPK,NIL,https://www.wikidata.org/wiki/Q8316772,https://www.wikidata.org/wiki/Q13091123,https://www.wikidata.org/wiki/Q13091123
497,de Lamarzelle,SPK,https://www.wikidata.org/wiki/Q3121364,https://www.wikidata.org/wiki/Q3121364,https://www.wikidata.org/wiki/Q3121364,NaN
2031,Armand Chouffet,SPK,https://www.wikidata.org/wiki/Q2861802,https://www.wikidata.org/wiki/Q2861802,NaN,NaN
4418,Lacombe,PER,https://www.wikidata.org/wiki/Q3385710,https://www.wikidata.org/wiki/Q3385710,https://www.wikidata.org/wiki/Q16025911,NaN


## Métrique - selection des candidats - Rappel@1 et Rappel@10

In [ ]:
# Colonnes des candidats (dans l'ordre de rang)
colonnes_candidats = [f"persons_{i}_wikidata_url" for i in range(1, 11)]
colonnes_candidats = [c for c in colonnes_candidats if c in df.columns]
print(f"Colonnes candidates trouvées : {colonnes_candidats}")

Colonnes candidates trouvées : ['persons_1_wikidata_url', 'persons_2_wikidata_url', 'persons_3_wikidata_url', 'persons_4_wikidata_url', 'persons_5_wikidata_url', 'persons_6_wikidata_url', 'persons_7_wikidata_url', 'persons_8_wikidata_url', 'persons_9_wikidata_url', 'persons_10_wikidata_url']


In [ ]:
# Extraction des QID : gold (vérité terrain) et candidats à chaque rang

def extraire_qid(url):
    """Extrait le QID (ex: Q12345) depuis une URL Wikidata."""
    if pd.isna(url) or not isinstance(url, str):
        return None
    match = re.search(r"(Q\d+)", url)
    return match.group(1) if match else None

df["qid_gold"] = df["wikidata_url_valide"].apply(extraire_qid)

for i, col in enumerate(colonnes_candidats, start=1):
    df[f"qid_candidat_{i}"] = df[col].apply(extraire_qid)

In [ ]:
# On évalue uniquement les lignes où une entité correcte existe (gold non vide/non NIL)
# -> le cas "pas d'entité correspondante" (NIL) relève d'une évaluation à part (NIL prediction),
#    pas du rappel de sélection de candidats.
df_eval = df[df["qid_gold"].notna()].copy()

print(f"Lignes évaluées (gold non vide) : {len(df_eval)} / {len(df)}")
print(f"Lignes exclues (gold vide/NIL)  : {len(df) - len(df_eval)}")

Lignes évaluées (gold non vide) : 4510 / 4686
Lignes exclues (gold vide/NIL)  : 176


In [ ]:
def rang_du_bon_candidat(row, colonnes_candidats):
    """Renvoie le rang (1-indexé) du premier candidat qui correspond au gold, sinon None."""
    for i, col in enumerate([f"qid_candidat_{k}" for k in range(1, len(colonnes_candidats) + 1)], start=1):
        if row[col] == row["qid_gold"]:
            return i
    return None

df_eval["rang_bon_candidat"] = df_eval.apply(rang_du_bon_candidat, args=(colonnes_candidats,), axis=1)

In [ ]:
def recall_at_k(df_eval, k):
    """Rappel@k : proportion de lignes où le bon candidat est trouvé dans les k premiers rangs."""
    trouve_dans_k = df_eval["rang_bon_candidat"].apply(lambda r: r is not None and r <= k)
    return trouve_dans_k.sum() / len(df_eval)

nb_candidats_max = len(colonnes_candidats)

recall_scores = {k: recall_at_k(df_eval, k) for k in range(1, nb_candidats_max + 1)}

print("Rappel@k :")
for k, score in recall_scores.items():
    print(f"  Rappel@{k:2d} : {score:.2%}")

Rappel@k :
  Rappel@ 1 : 77.63%
  Rappel@ 2 : 83.33%
  Rappel@ 3 : 86.45%
  Rappel@ 4 : 89.45%
  Rappel@ 5 : 90.40%
  Rappel@ 6 : 90.86%
  Rappel@ 7 : 91.00%
  Rappel@ 8 : 91.02%
  Rappel@ 9 : 91.77%
  Rappel@10 : 91.77%


In [ ]:
# Focus sur les deux métriques principales demandées
print(f"Rappel@1  : {recall_scores[1]:.2%}")
print(f"Rappel@{nb_candidats_max} : {recall_scores[nb_candidats_max]:.2%}")

# Cas où le bon candidat n'apparaît dans aucun des rangs proposés
non_trouve = df_eval["rang_bon_candidat"].isna().sum()
print(f"\nBon candidat absent des {nb_candidats_max} proposés : {non_trouve} / {len(df_eval)} "
      f"({non_trouve/len(df_eval):.2%})")

Rappel@1  : 77.63%
Rappel@10 : 91.77%

Bon candidat absent des 10 proposés : 371 / 4510 (8.23%)


In [ ]:
# Distribution des rangs auxquels le bon candidat est trouvé (utile pour visualiser où ça "coince")
distribution_rangs = df_eval["rang_bon_candidat"].value_counts(dropna=False).sort_index()
print("Distribution des rangs du bon candidat (NaN = jamais trouvé) :")
print(distribution_rangs)

Distribution des rangs du bon candidat (NaN = jamais trouvé) :
rang_bon_candidat
1.0    3501
2.0     257
3.0     141
4.0     135
5.0      43
6.0      21
7.0       6
8.0       1
9.0      34
NaN     371
Name: count, dtype: int64


In [ ]:
def recall_at_k_by_group(df_eval, group_col, k):
    return (
        df_eval.groupby(group_col, dropna=False)["rang_bon_candidat"]
        .apply(lambda s: (s.dropna() <= k).sum() / len(s))
        .rename(f"rappel@{k}")
    )

recap_recall = pd.DataFrame({
    "rappel@1": recall_at_k_by_group(df_eval, "classe", 1),
    f"rappel@{nb_candidats_max}": recall_at_k_by_group(df_eval, "classe", nb_candidats_max),
    "nb_lignes": df_eval.groupby("classe", dropna=False).size(),
})
recap_recall

,rappel@1,rappel@10,nb_lignes
classe,,,
PER,0.857934,0.932841,2710
SPK,0.653333,0.895000,1800


## Métrique - Désambiguisation - Accuracy@1

In [ ]:
# Accuracy@1 : proportion de mentions pour lesquelles le candidat classé 1er
# correspond exactement à la bonne entité (gold)
df_eval["candidat_1_correct"] = df_eval["qid_candidat_1"] == df_eval["qid_gold"]

accuracy_at_1 = df_eval["candidat_1_correct"].sum() / len(df_eval)

print(f"Accuracy@1 : {accuracy_at_1:.2%} "
      f"({df_eval['candidat_1_correct'].sum()} / {len(df_eval)})")

Accuracy@1 : 77.63% (3501 / 4510)


In [ ]:
# Ventilation par classe
accuracy_at_1_par_classe = (
    df_eval.groupby("classe", dropna=False)["candidat_1_correct"]
    .agg(nb_correct="sum", nb_total="count")
)
accuracy_at_1_par_classe["accuracy@1"] = (
    accuracy_at_1_par_classe["nb_correct"] / accuracy_at_1_par_classe["nb_total"]
).round(4)

accuracy_at_1_par_classe

,nb_correct,nb_total,accuracy@1
classe,,,
PER,2325,2710,0.8579
SPK,1176,1800,0.6533


## Métrique - Selection des candidats - NIL detection

Système prédit NIL = aucune des colonnes persons_i_wikidata_url n'est renseignée pour la mention.

Vrai NIL : le gold est vide et le système n'a rien proposé → comportement correct.

Silence (faux NIL) : une entité correcte existait, mais le système n'a proposé aucun candidat — c'est l'échec le plus problématique pour le rappel global, car il n'est pas rattrapable même en augmentant k.

Faux positif NIL : aucune entité correcte n'existait, mais le système a quand même proposé un ou plusieurs candidats — ces cas devraient normalement être filtrés en aval par la validation humaine (check_validity).

In [ ]:
# --- Préparation ---
# On repart du df complet (les lignes gold vides sont ici essentielles, on ne les exclut plus)
df["qid_gold"] = df["wikidata_url_valide"].apply(extraire_qid)

for i, col in enumerate(colonnes_candidats, start=1):
    df[f"qid_candidat_{i}"] = df[col].apply(extraire_qid)

colonnes_qid_candidats = [f"qid_candidat_{i}" for i in range(1, len(colonnes_candidats) + 1)]

# Le système a-t-il proposé au moins un candidat (non vide) pour cette mention ?
df["systeme_propose_candidat"] = df[colonnes_qid_candidats].notna().any(axis=1)

# Prédiction NIL du système = aucun candidat proposé
df["systeme_predit_nil"] = ~df["systeme_propose_candidat"]

# Gold NIL = pas d'entité correcte attendue
df["gold_est_nil"] = df["qid_gold"].isna()

In [ ]:
# --- Matrice de confusion NIL vs non-NIL ---
vrai_nil = (df["gold_est_nil"] & df["systeme_predit_nil"]).sum()              # correct : rien à trouver, rien proposé
faux_nil = (~df["gold_est_nil"] & df["systeme_predit_nil"]).sum()             # erreur : entité existait, rien proposé (silence)
faux_non_nil = (df["gold_est_nil"] & ~df["systeme_predit_nil"]).sum()         # erreur : rien à trouver, mais candidat(s) proposé(s)
vrai_non_nil = (~df["gold_est_nil"] & ~df["systeme_predit_nil"]).sum()        # cas normal : entité existait, candidat(s) proposé(s)

matrice_nil = pd.DataFrame({
    "gold = NIL": [vrai_nil, faux_non_nil],
    "gold = entité": [faux_nil, vrai_non_nil],
}, index=["système prédit NIL", "système propose ≥1 candidat"])

print(matrice_nil)
print(f"\nTotal : {len(df)} = {vrai_nil + faux_nil + faux_non_nil + vrai_non_nil}")

                             gold = NIL  gold = entité
système prédit NIL                   83            114
système propose ≥1 candidat          93           4396

Total : 4686 = 4686


In [ ]:
# --- Métriques NIL ---
nb_gold_nil = df["gold_est_nil"].sum()
nb_gold_entite = (~df["gold_est_nil"]).sum()

# Précision NIL : parmi les prédictions NIL du système, combien sont correctes (gold aussi NIL) ?
nb_predictions_nil = df["systeme_predit_nil"].sum()
precision_nil = vrai_nil / nb_predictions_nil if nb_predictions_nil else float("nan")

# Rappel NIL : parmi les cas où gold = NIL, combien le système a-t-il bien détectés comme NIL ?
rappel_nil = vrai_nil / nb_gold_nil if nb_gold_nil else float("nan")

# Taux de "silence" : le système ne propose rien alors qu'une entité correcte existait
taux_silence = faux_nil / nb_gold_entite if nb_gold_entite else float("nan")

# F1 NIL
f1_nil = (
    2 * precision_nil * rappel_nil / (precision_nil + rappel_nil)
    if (precision_nil + rappel_nil) > 0 else float("nan")
)

print(f"Précision NIL     : {precision_nil:.2%}  ({vrai_nil} / {nb_predictions_nil})")
print(f"Rappel NIL        : {rappel_nil:.2%}  ({vrai_nil} / {nb_gold_nil})")
print(f"F1 NIL            : {f1_nil:.2%}")
print(f"Taux de silence   : {taux_silence:.2%}  ({faux_nil} / {nb_gold_entite})  "
      f"— entité existait mais aucun candidat proposé")
print(f"Faux positifs NIL : {faux_non_nil} / {nb_gold_nil}  "
      f"— aucune entité attendue mais candidat(s) proposé(s)")

Précision NIL     : 42.13%  (83 / 197)
Rappel NIL        : 47.16%  (83 / 176)
F1 NIL            : 44.50%
Taux de silence   : 2.53%  (114 / 4510)  — entité existait mais aucun candidat proposé
Faux positifs NIL : 93 / 176  — aucune entité attendue mais candidat(s) proposé(s)


In [ ]:
# --- Ventilation par classe ---
def stats_nil_par_groupe(df, group_col):
    resultats = []
    for val, sous_df in df.groupby(group_col, dropna=False):
        vn = (sous_df["gold_est_nil"] & sous_df["systeme_predit_nil"]).sum()
        fn = (~sous_df["gold_est_nil"] & sous_df["systeme_predit_nil"]).sum()
        fnn = (sous_df["gold_est_nil"] & ~sous_df["systeme_predit_nil"]).sum()
        n_gold_nil = sous_df["gold_est_nil"].sum()
        n_gold_ent = (~sous_df["gold_est_nil"]).sum()
        resultats.append({
            group_col: val,
            "nb_lignes": len(sous_df),
            "rappel_nil": vn / n_gold_nil if n_gold_nil else np.nan,
            "taux_silence": fn / n_gold_ent if n_gold_ent else np.nan,
            "faux_positifs_nil": fnn,
        })
    return pd.DataFrame(resultats)

stats_nil_par_groupe(df, "classe")

,classe,nb_lignes,rappel_nil,taux_silence,faux_positifs_nil
0,PER,2877,0.491018,0.025092,85
1,SPK,1809,0.111111,0.025556,8


## Inspection d'exemples

In [ ]:
# Taille des échantillons à afficher (ajustable)
TAILLE_ECHANTILLON = 10
RANDOM_STATE = 42

colonnes_affichage_base = ["entity", "classe", "wikidata_url_valide"]
colonnes_affichage_candidats = [c for c in colonnes_candidats if c in df.columns]
colonnes_affichage = [c for c in colonnes_affichage_base + colonnes_affichage_candidats if c in df.columns]

Échantillons pour le rappel (Recall@1 / Recall@k)

In [ ]:
# Trouvé en position 1
ech_rang1 = df_eval[df_eval["rang_bon_candidat"] == 1].sample(
    min(TAILLE_ECHANTILLON, (df_eval["rang_bon_candidat"] == 1).sum()), random_state=RANDOM_STATE
)
print(f"=== Échantillon : bon candidat trouvé en rang 1 ({(df_eval['rang_bon_candidat'] == 1).sum()} au total) ===")
display(ech_rang1[colonnes_affichage + ["rang_bon_candidat"]])

=== Échantillon : bon candidat trouvé en rang 1 (3501 au total) ===


,entity,classe,wikidata_url_valide,persons_1_wikidata_url,persons_2_wikidata_url,persons_3_wikidata_url,persons_4_wikidata_url,persons_5_wikidata_url,persons_6_wikidata_url,persons_7_wikidata_url,persons_8_wikidata_url,persons_9_wikidata_url,persons_10_wikidata_url,rang_bon_candidat
369,Freppel,SPK,https://www.wikidata.org/wiki/Q1063452,https://www.wikidata.org/wiki/Q1063452,https://www.wikidata.org/wiki/Q1063452,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
1888,le ministre des finances,PER,https://www.wikidata.org/wiki/Q3132698,https://www.wikidata.org/wiki/Q3132698,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
2843,le ministre des finances,SPK,https://www.wikidata.org/wiki/Q3132698,https://www.wikidata.org/wiki/Q3132698,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
1204,Fouquier (Henry),PER,https://www.wikidata.org/wiki/Q3132781,https://www.wikidata.org/wiki/Q3132781,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
1215,Gerville-Réache,PER,https://www.wikidata.org/wiki/Q3099023,https://www.wikidata.org/wiki/Q3099023,https://www.wikidata.org/wiki/Q3099023,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
3657,Defarge,PER,https://www.wikidata.org/wiki/Q2824801,https://www.wikidata.org/wiki/Q2824801,https://www.wikidata.org/wiki/Q2824801,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
1353,Roche (Jules) (Savoie),PER,https://www.wikidata.org/wiki/Q3188891,https://www.wikidata.org/wiki/Q3188891,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
4123,Vincent Auriol,SPK,https://www.wikidata.org/wiki/Q158772,https://www.wikidata.org/wiki/Q158772,https://www.wikidata.org/wiki/Q158772,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
4525,le ministre de la guerre d'alors,PER,https://www.wikidata.org/wiki/Q317957,https://www.wikidata.org/wiki/Q317957,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
3923,Jean Bosc,PER,https://www.wikidata.org/wiki/Q1685388,https://www.wikidata.org/wiki/Q1685388,https://www.wikidata.org/wiki/Q1685388,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0


In [ ]:
# Trouvé mais pas en 1ère position (rang 2 à k) — cas intéressants : le re-ranking a échoué
mask_rang_tardif = df_eval["rang_bon_candidat"] > 1
ech_rang_tardif = df_eval[mask_rang_tardif].sample(
    min(TAILLE_ECHANTILLON, mask_rang_tardif.sum()), random_state=RANDOM_STATE
)
print(f"=== Échantillon : bon candidat trouvé mais pas en 1er ({mask_rang_tardif.sum()} au total) ===")
display(ech_rang_tardif[colonnes_affichage + ["rang_bon_candidat"]])

=== Échantillon : bon candidat trouvé mais pas en 1er (638 au total) ===


,entity,classe,wikidata_url_valide,persons_1_wikidata_url,persons_2_wikidata_url,persons_3_wikidata_url,persons_4_wikidata_url,persons_5_wikidata_url,persons_6_wikidata_url,persons_7_wikidata_url,persons_8_wikidata_url,persons_9_wikidata_url,persons_10_wikidata_url,rang_bon_candidat
1827,le président,SPK,https://www.wikidata.org/wiki/Q459510,https://www.wikidata.org/wiki/Q346648,https://www.wikidata.org/wiki/Q2117075,https://www.wikidata.org/wiki/Q12680,https://www.wikidata.org/wiki/Q459510,NaN,NaN,NaN,NaN,NaN,NaN,4.0
1697,le ministre,PER,https://www.wikidata.org/wiki/Q2978569,https://www.wikidata.org/wiki/Q554679,https://www.wikidata.org/wiki/Q3219208,https://www.wikidata.org/wiki/Q346648,https://www.wikidata.org/wiki/Q3085590,https://www.wikidata.org/wiki/Q3132698,https://www.wikidata.org/wiki/Q2978569,https://www.wikidata.org/wiki/Q318617,https://www.wikidata.org/wiki/Q2117075,https://www.wikidata.org/wiki/Q728806,https://www.wikidata.org/wiki/Q179888,6.0
214,le président,SPK,https://www.wikidata.org/wiki/Q471311,https://www.wikidata.org/wiki/Q317957,https://www.wikidata.org/wiki/Q585025,https://www.wikidata.org/wiki/Q471311,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0
4273,le président,SPK,https://www.wikidata.org/wiki/Q585025,https://www.wikidata.org/wiki/Q708359,https://www.wikidata.org/wiki/Q585025,https://www.wikidata.org/wiki/Q459914,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0
4485,le président,SPK,https://www.wikidata.org/wiki/Q585025,https://www.wikidata.org/wiki/Q708359,https://www.wikidata.org/wiki/Q585025,https://www.wikidata.org/wiki/Q459914,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0
213,le comte de Maillé,SPK,https://www.wikidata.org/wiki/Q2861743,https://www.wikidata.org/wiki/Q5624188,https://www.wikidata.org/wiki/Q2861743,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0
4389,le président,SPK,https://www.wikidata.org/wiki/Q585025,https://www.wikidata.org/wiki/Q708359,https://www.wikidata.org/wiki/Q585025,https://www.wikidata.org/wiki/Q459914,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0
640,le président,SPK,https://www.wikidata.org/wiki/Q471311,https://www.wikidata.org/wiki/Q317957,https://www.wikidata.org/wiki/Q585025,https://www.wikidata.org/wiki/Q471311,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0
1768,le président,SPK,https://www.wikidata.org/wiki/Q459510,https://www.wikidata.org/wiki/Q346648,https://www.wikidata.org/wiki/Q2117075,https://www.wikidata.org/wiki/Q12680,https://www.wikidata.org/wiki/Q459510,NaN,NaN,NaN,NaN,NaN,NaN,4.0
3981,le président,SPK,https://www.wikidata.org/wiki/Q3188688,https://www.wikidata.org/wiki/Q18434,https://www.wikidata.org/wiki/Q213526,https://www.wikidata.org/wiki/Q3188688,https://www.wikidata.org/wiki/Q274344,NaN,NaN,NaN,NaN,NaN,NaN,3.0


In [ ]:
# Jamais trouvé parmi les k candidats — échec de la génération de candidats elle-même
mask_non_trouve = df_eval["rang_bon_candidat"].isna()
ech_non_trouve = df_eval[mask_non_trouve].sample(
    min(TAILLE_ECHANTILLON, mask_non_trouve.sum()), random_state=RANDOM_STATE
)
print(f"=== Échantillon : bon candidat absent des {nb_candidats_max} proposés ({mask_non_trouve.sum()} au total) ===")
display(ech_non_trouve[colonnes_affichage])

=== Échantillon : bon candidat absent des 10 proposés (371 au total) ===


,entity,classe,wikidata_url_valide,persons_1_wikidata_url,persons_2_wikidata_url,persons_3_wikidata_url,persons_4_wikidata_url,persons_5_wikidata_url,persons_6_wikidata_url,persons_7_wikidata_url,persons_8_wikidata_url,persons_9_wikidata_url,persons_10_wikidata_url
4167,le président de la commission des finances,PER,https://www.wikidata.org/wiki/Q450463,https://www.wikidata.org/wiki/Q18434,https://www.wikidata.org/wiki/Q213526,https://www.wikidata.org/wiki/Q3188688,https://www.wikidata.org/wiki/Q274344,NaN,NaN,NaN,NaN,NaN,NaN
1479,le rapporteur,PER,https://www.wikidata.org/wiki/Q2899637,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
566,le ministre,PER,https://www.wikidata.org/wiki/Q16025808,https://www.wikidata.org/wiki/Q939193,https://www.wikidata.org/wiki/Q1934639,https://www.wikidata.org/wiki/Q471293,https://www.wikidata.org/wiki/Q317957,https://www.wikidata.org/wiki/Q296673,https://www.wikidata.org/wiki/Q1857743,https://www.wikidata.org/wiki/Q358387,https://www.wikidata.org/wiki/Q249982,https://www.wikidata.org/wiki/Q217706,https://www.wikidata.org/wiki/Q3188891
4022,le président de la commission de l'administrat...,PER,https://www.wikidata.org/wiki/Q450463,https://www.wikidata.org/wiki/Q18434,https://www.wikidata.org/wiki/Q213526,https://www.wikidata.org/wiki/Q3188688,https://www.wikidata.org/wiki/Q274344,NaN,NaN,NaN,NaN,NaN,NaN
2268,le président,PER,https://www.wikidata.org/wiki/Q3143189,https://www.wikidata.org/wiki/Q346648,https://www.wikidata.org/wiki/Q2117075,https://www.wikidata.org/wiki/Q12680,https://www.wikidata.org/wiki/Q459510,NaN,NaN,NaN,NaN,NaN,NaN
3271,ministre,PER,https://www.wikidata.org/wiki/Q3272346,https://www.wikidata.org/wiki/Q3271218,https://www.wikidata.org/wiki/Q296064,https://www.wikidata.org/wiki/Q471293,https://www.wikidata.org/wiki/Q3300415,https://www.wikidata.org/wiki/Q3261042,https://www.wikidata.org/wiki/Q3057194,https://www.wikidata.org/wiki/Q2862434,https://www.wikidata.org/wiki/Q503827,https://www.wikidata.org/wiki/Q3588648,https://www.wikidata.org/wiki/Q275020
2370,le président,SPK,https://www.wikidata.org/wiki/Q3143189,https://www.wikidata.org/wiki/Q346648,https://www.wikidata.org/wiki/Q2117075,https://www.wikidata.org/wiki/Q12680,https://www.wikidata.org/wiki/Q459510,NaN,NaN,NaN,NaN,NaN,NaN
2570,le président,SPK,https://www.wikidata.org/wiki/Q3143189,https://www.wikidata.org/wiki/Q346648,https://www.wikidata.org/wiki/Q2117075,https://www.wikidata.org/wiki/Q12680,https://www.wikidata.org/wiki/Q459510,NaN,NaN,NaN,NaN,NaN,NaN
3815,Roger,PER,https://www.wikidata.org/wiki/Q3174388,https://www.wikidata.org/wiki/Q3170240,https://www.wikidata.org/wiki/Q3170240,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2617,le président,SPK,https://www.wikidata.org/wiki/Q3143189,https://www.wikidata.org/wiki/Q346648,https://www.wikidata.org/wiki/Q2117075,https://www.wikidata.org/wiki/Q12680,https://www.wikidata.org/wiki/Q459510,NaN,NaN,NaN,NaN,NaN,NaN


Échantillons pour Accuracy@1

In [ ]:
# Candidat 1 correct
ech_acc1_correct = df_eval[df_eval["candidat_1_correct"]].sample(
    min(TAILLE_ECHANTILLON, df_eval["candidat_1_correct"].sum()), random_state=RANDOM_STATE
)
print(f"=== Échantillon : candidat 1 correct ({df_eval['candidat_1_correct'].sum()} au total) ===")
display(ech_acc1_correct[colonnes_affichage])

=== Échantillon : candidat 1 correct (3501 au total) ===


,entity,classe,wikidata_url_valide,persons_1_wikidata_url,persons_2_wikidata_url,persons_3_wikidata_url,persons_4_wikidata_url,persons_5_wikidata_url,persons_6_wikidata_url,persons_7_wikidata_url,persons_8_wikidata_url,persons_9_wikidata_url,persons_10_wikidata_url
369,Freppel,SPK,https://www.wikidata.org/wiki/Q1063452,https://www.wikidata.org/wiki/Q1063452,https://www.wikidata.org/wiki/Q1063452,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1888,le ministre des finances,PER,https://www.wikidata.org/wiki/Q3132698,https://www.wikidata.org/wiki/Q3132698,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2843,le ministre des finances,SPK,https://www.wikidata.org/wiki/Q3132698,https://www.wikidata.org/wiki/Q3132698,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1204,Fouquier (Henry),PER,https://www.wikidata.org/wiki/Q3132781,https://www.wikidata.org/wiki/Q3132781,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1215,Gerville-Réache,PER,https://www.wikidata.org/wiki/Q3099023,https://www.wikidata.org/wiki/Q3099023,https://www.wikidata.org/wiki/Q3099023,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3657,Defarge,PER,https://www.wikidata.org/wiki/Q2824801,https://www.wikidata.org/wiki/Q2824801,https://www.wikidata.org/wiki/Q2824801,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1353,Roche (Jules) (Savoie),PER,https://www.wikidata.org/wiki/Q3188891,https://www.wikidata.org/wiki/Q3188891,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4123,Vincent Auriol,SPK,https://www.wikidata.org/wiki/Q158772,https://www.wikidata.org/wiki/Q158772,https://www.wikidata.org/wiki/Q158772,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4525,le ministre de la guerre d'alors,PER,https://www.wikidata.org/wiki/Q317957,https://www.wikidata.org/wiki/Q317957,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3923,Jean Bosc,PER,https://www.wikidata.org/wiki/Q1685388,https://www.wikidata.org/wiki/Q1685388,https://www.wikidata.org/wiki/Q1685388,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Candidat 1 incorrect (mais un candidat a bien été proposé)
mask_acc1_incorrect = (~df_eval["candidat_1_correct"]) & df_eval["qid_candidat_1"].notna()
ech_acc1_incorrect = df_eval[mask_acc1_incorrect].sample(
    min(TAILLE_ECHANTILLON, mask_acc1_incorrect.sum()), random_state=RANDOM_STATE
)
print(f"=== Échantillon : candidat 1 incorrect ({mask_acc1_incorrect.sum()} au total) ===")
display(ech_acc1_incorrect[colonnes_affichage])

=== Échantillon : candidat 1 incorrect (895 au total) ===


,entity,classe,wikidata_url_valide,persons_1_wikidata_url,persons_2_wikidata_url,persons_3_wikidata_url,persons_4_wikidata_url,persons_5_wikidata_url,persons_6_wikidata_url,persons_7_wikidata_url,persons_8_wikidata_url,persons_9_wikidata_url,persons_10_wikidata_url
3861,le président,SPK,https://www.wikidata.org/wiki/Q296673,https://www.wikidata.org/wiki/Q358452,https://www.wikidata.org/wiki/Q296673,https://www.wikidata.org/wiki/Q309995,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2612,le président,SPK,https://www.wikidata.org/wiki/Q3143189,https://www.wikidata.org/wiki/Q346648,https://www.wikidata.org/wiki/Q2117075,https://www.wikidata.org/wiki/Q12680,https://www.wikidata.org/wiki/Q459510,NaN,NaN,NaN,NaN,NaN,NaN
3005,Béranger,PER,https://www.wikidata.org/wiki/Q3425950,https://www.wikidata.org/wiki/Q3384252,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3879,le président,SPK,https://www.wikidata.org/wiki/Q296673,https://www.wikidata.org/wiki/Q358452,https://www.wikidata.org/wiki/Q296673,https://www.wikidata.org/wiki/Q309995,NaN,NaN,NaN,NaN,NaN,NaN,NaN
227,le ministre,SPK,https://www.wikidata.org/wiki/Q1934639,https://www.wikidata.org/wiki/Q939193,https://www.wikidata.org/wiki/Q1934639,https://www.wikidata.org/wiki/Q471293,https://www.wikidata.org/wiki/Q317957,https://www.wikidata.org/wiki/Q296673,https://www.wikidata.org/wiki/Q1857743,https://www.wikidata.org/wiki/Q358387,https://www.wikidata.org/wiki/Q249982,https://www.wikidata.org/wiki/Q217706,https://www.wikidata.org/wiki/Q3188891
1889,le président,SPK,https://www.wikidata.org/wiki/Q459510,https://www.wikidata.org/wiki/Q346648,https://www.wikidata.org/wiki/Q2117075,https://www.wikidata.org/wiki/Q12680,https://www.wikidata.org/wiki/Q459510,NaN,NaN,NaN,NaN,NaN,NaN
1938,le président,SPK,https://www.wikidata.org/wiki/Q459510,https://www.wikidata.org/wiki/Q346648,https://www.wikidata.org/wiki/Q2117075,https://www.wikidata.org/wiki/Q12680,https://www.wikidata.org/wiki/Q459510,NaN,NaN,NaN,NaN,NaN,NaN
2054,le président,SPK,https://www.wikidata.org/wiki/Q459510,https://www.wikidata.org/wiki/Q346648,https://www.wikidata.org/wiki/Q2117075,https://www.wikidata.org/wiki/Q12680,https://www.wikidata.org/wiki/Q459510,NaN,NaN,NaN,NaN,NaN,NaN
1564,le ministre,PER,https://www.wikidata.org/wiki/Q3132698,https://www.wikidata.org/wiki/Q554679,https://www.wikidata.org/wiki/Q3219208,https://www.wikidata.org/wiki/Q346648,https://www.wikidata.org/wiki/Q3085590,https://www.wikidata.org/wiki/Q3132698,https://www.wikidata.org/wiki/Q2978569,https://www.wikidata.org/wiki/Q318617,https://www.wikidata.org/wiki/Q2117075,https://www.wikidata.org/wiki/Q728806,https://www.wikidata.org/wiki/Q179888
774,le président,PER,https://www.wikidata.org/wiki/Q471311,https://www.wikidata.org/wiki/Q317957,https://www.wikidata.org/wiki/Q585025,https://www.wikidata.org/wiki/Q471311,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Échantillons pour la prédiction NIL

In [ ]:
# Vrai NIL : gold vide, système n'a rien proposé (comportement correct)
mask_vrai_nil = df["gold_est_nil"] & df["systeme_predit_nil"]
ech_vrai_nil = df[mask_vrai_nil].sample(
    min(TAILLE_ECHANTILLON, mask_vrai_nil.sum()), random_state=RANDOM_STATE
)
print(f"=== Échantillon : vrai NIL ({mask_vrai_nil.sum()} au total) ===")
display(ech_vrai_nil[colonnes_affichage])

=== Échantillon : vrai NIL (83 au total) ===


,entity,classe,wikidata_url_valide,persons_1_wikidata_url,persons_2_wikidata_url,persons_3_wikidata_url,persons_4_wikidata_url,persons_5_wikidata_url,persons_6_wikidata_url,persons_7_wikidata_url,persons_8_wikidata_url,persons_9_wikidata_url,persons_10_wikidata_url
194,l'inspecteur primaire,PER,NIL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
99,la sœur supé- rieuse,PER,NIL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
186,Le préfet,PER,NIL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
201,l'inspecteur,PER,NIL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
166,le maire de Vicq,PER,NIL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
192,L'inspecteur primaire,PER,NIL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
121,l'institutrice congréganiste,PER,NIL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
263,Femme Recouvreur,PER,NIL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
105,l'inspecteur d'académie,PER,NIL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
137,"Le directeur, « POUZET",PER,NIL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Silence (faux NIL) : entité correcte existait, système n'a rien proposé — erreur la plus coûteuse
mask_silence = ~df["gold_est_nil"] & df["systeme_predit_nil"]
ech_silence = df[mask_silence].sample(
    min(TAILLE_ECHANTILLON, mask_silence.sum()), random_state=RANDOM_STATE
)
print(f"=== Échantillon : silence, entité manquée ({mask_silence.sum()} au total) ===")
display(ech_silence[colonnes_affichage])

=== Échantillon : silence, entité manquée (114 au total) ===


,entity,classe,wikidata_url_valide,persons_1_wikidata_url,persons_2_wikidata_url,persons_3_wikidata_url,persons_4_wikidata_url,persons_5_wikidata_url,persons_6_wikidata_url,persons_7_wikidata_url,persons_8_wikidata_url,persons_9_wikidata_url,persons_10_wikidata_url
4032,rapporteur général,PER,https://www.wikidata.org/wiki/Q2821514,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
428,L'honorable évêque d'Angers,PER,https://www.wikidata.org/wiki/Q1063452,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2517,rapporteur de la commission,PER,https://www.wikidata.org/wiki/Q2899637,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3277,le roi de Tunis,PER,https://www.wikidata.org/wiki/Q1951858,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
905,Boissel (de),PER,https://www.wikidata.org/wiki/Q2960523,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2651,le rapporteur,SPK,https://www.wikidata.org/wiki/Q2899637,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3278,Le roi de Tunis,PER,https://www.wikidata.org/wiki/Q1951858,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3269,Le bey de Tunis,PER,https://www.wikidata.org/wiki/Q2551816,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2683,le rapporteur,PER,https://www.wikidata.org/wiki/Q2899637,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
937,Du Boudan,PER,https://www.wikidata.org/wiki/Q2959275,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Faux positif NIL : rien à trouver, mais système a proposé un/des candidat(s)
mask_faux_pos_nil = df["gold_est_nil"] & ~df["systeme_predit_nil"]
ech_faux_pos_nil = df[mask_faux_pos_nil].sample(
    min(TAILLE_ECHANTILLON, mask_faux_pos_nil.sum()), random_state=RANDOM_STATE
)
print(f"=== Échantillon : faux positif NIL ({mask_faux_pos_nil.sum()} au total) ===")
display(ech_faux_pos_nil[colonnes_affichage])

=== Échantillon : faux positif NIL (93 au total) ===


,entity,classe,wikidata_url_valide,persons_1_wikidata_url,persons_2_wikidata_url,persons_3_wikidata_url,persons_4_wikidata_url,persons_5_wikidata_url,persons_6_wikidata_url,persons_7_wikidata_url,persons_8_wikidata_url,persons_9_wikidata_url,persons_10_wikidata_url
1086,"Vileneuve (marquis de), Witt (Conrad de)",PER,NIL,https://www.wikidata.org/wiki/Q3131576,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
252,Rouger,PER,NIL,https://www.wikidata.org/wiki/Q29767,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2413,Emile Faure,SPK,NIL,https://www.wikidata.org/wiki/Q8316772,https://www.wikidata.org/wiki/Q13091123,https://www.wikidata.org/wiki/Q13091123,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3452,le président de la commission,SPK,NIL,https://www.wikidata.org/wiki/Q275020,https://www.wikidata.org/wiki/Q296673,https://www.wikidata.org/wiki/Q669725,NaN,NaN,NaN,NaN,NaN,NaN,NaN
34,le marquis de La Ferrière,PER,NIL,https://www.wikidata.org/wiki/Q3131576,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
364,le préfet de la Haute-Marne,PER,NIL,https://www.wikidata.org/wiki/Q2602696,https://www.wikidata.org/wiki/Q3570605,https://www.wikidata.org/wiki/Q3008869,https://www.wikidata.org/wiki/Q16025675,https://www.wikidata.org/wiki/Q16025951,https://www.wikidata.org/wiki/Q2411428,https://www.wikidata.org/wiki/Q3132908,https://www.wikidata.org/wiki/Q2861822,https://www.wikidata.org/wiki/Q3104108,https://www.wikidata.org/wiki/Q15973218
1054,"Prax-Paris, Prémat",PER,NIL,https://www.wikidata.org/wiki/Q766851,https://www.wikidata.org/wiki/Q15969647,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2975,le percepteur de Saint-Quentin,PER,NIL,https://www.wikidata.org/wiki/Q2959090,https://www.wikidata.org/wiki/Q3557652,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
120,le maire,PER,NIL,https://www.wikidata.org/wiki/Q7183006,https://www.wikidata.org/wiki/Q3132907,https://www.wikidata.org/wiki/Q3103051,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1583,Valentino,PER,NIL,https://www.wikidata.org/wiki/Q16026578,https://www.wikidata.org/wiki/Q3085956,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
